<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/secondWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **makemore = character level language model**

first week me micrograd banaya tha.... ab hum karpathy ke makemore ko follow karenge.

step by step chalenge -> bigram -> bigram as neural net -> mlp with character embeddings -> weight init -> batch norm.

dataset = 32k naam (names.txt).

In [ ]:
# first we download karpathy ka names.txt....
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')

words = open('names.txt').read().splitlines()
print('total names:', len(words))
print(words[:8])

# **bigram model = just counting**

idea bahut simple hai.... har character ke baad kaunsa character aata hai bas uski ginti kar lo. '.' start aur end ka special token hai.

In [ ]:
import torch

# now we make the vocabulary.... char to index and index to char
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                       # '.' ko index 0 diya
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('vocab size:', vocab_size)

# 27x27 count matrix.... which char comes after which
N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

In [ ]:
# now we convert the counts into probability.... +1 is smoothing so no zero prob
P = (N + 1).float()
P /= P.sum(1, keepdim=True)          # har row ka sum 1 ho jaye

# ab hum is bigram model se kuch naam sample karte hai
g = torch.Generator().manual_seed(2147483647)
for _ in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

In [ ]:
# how good is the model.... we check using negative log likelihood (lower is better)
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob)
        n += 1
print('nll:', (-log_likelihood / n).item())

# **same bigram but as a neural net**

ab wahi bigram hum ek neural net se seekhenge.... input char ko one-hot banao, ek linear layer W se multiply, phir softmax. end me loss almost same aata hai (jaisa hona chahiye).

In [ ]:
import torch.nn.functional as F

# training set banate hai.... x = current char, y = next char
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# gradient descent loop.... same idea as micrograd -> forward, backward, update
for k in range(120):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W                       # log-counts jaisa
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)   # softmax
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()

    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

print('final loss:', loss.item())
# ye almost bigram counts jitna hi aata hai.... matlab net ne wahi seekh liya

# **MLP model = Bengio 2003 paper**

bigram me hum sirf 1 previous char dekhte the.

Ab context badhate hai -> block_size = 3 previous chars.

Aur har char ko ek chhoti vector me convert karenge -> yahi character embedding hai (lookup table C).

dataset ko 80/10/10 me todte hai (train / dev / test).

In [ ]:
block_size = 3      # how many previous chars we look at


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # start me sab '.'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]     # sliding window aage khiskao
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print('train:', Xtr.shape, 'dev:', Xdev.shape, 'test:', Xte.shape)

# **weight init + batch normalization**

do important cheez jo makemore part 3 me sikhaya:

1. weight init -> agar weights bade random se start ho to starting loss bahut high aata hai aur tanh saturate (dead) ho jata hai. isliye W1 ko kaiming init (5/3 / sqrt(fan_in)) aur W2 ko chhota (*0.01) rakhte hai.

2. batch norm -> hidden pre-activation ko har batch me normalize kar dete hai (mean 0, std 1).... training smooth ho jati hai. running mean/std bhi rakhte hai inference ke time ke liye.

In [ ]:
n_embd = 10        # embedding size per char
n_hidden = 200     # hidden layer size
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
# kaiming init.... tanh gain 5/3 divided by sqrt(fan_in)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01   # chhota -> starting loss kam
b2 = torch.randn(vocab_size, generator=g) * 0

# batch norm ke parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print('total params:', sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# training loop (minibatch).... same forward -> backward -> update pattern
max_steps = 20000
batch_size = 32

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]                              # yaha character embedding lookup ho raha hai
    embcat = emb.view(emb.shape[0], -1)      # flatten
    hpreact = embcat @ W1 + b1

    # batch norm
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update.... learning rate ko baad me kam kar dete hai
    lr = 0.1 if i < 10000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 2000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

In [ ]:
# train aur dev pe loss.... yaha batch norm ke running stats use karte hai
@torch.no_grad()
def split_loss(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    print(split, F.cross_entropy(logits, Y).item())


split_loss('train')
split_loss('dev')

# **naam generate karo (mlp se)**

ab dekhte hai bigram wale se kitne behtar naam aate hai.

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(15):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view(1, -1)
        hpreact = embcat @ W1 + b1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

# **week 2 ka summary (mera notes)**

- bigram counts -> sabse simple model, nll around 2.45
- wahi bigram neural net se bhi ban gaya (same loss) -> gradient descent kaam karta hai
- mlp + character embeddings se context use kar paye, loss ~2.1 ke aas paas
- galat weight init se starting loss high aata hai aur tanh dead ho jata hai -> kaiming init se theek
- batch norm ne training smooth ki

next week -> rnn, lstm, attention, transformers....